# Maschinelles Lernen
## Übung Termin 1 (04.05.2026) – Datenaufbereitung

### Aufbau der Übung

**1. Importieren und Zusammenführen von Daten**
- Datensätze einladen
- Datensätze zusammenführen
- Relevante Variablen und Beobachtungen auswählen

**2. Datenaufbereitung**
- Werte unterhalb der Nachweisgrenze behandeln
- Fehlende Werte in numerischen und kategorischen Variablen ersetzen
- Features aggregieren
- Daten in Trainings- und Testdaten aufteilen
- Numerische Variablen skalieren bzw. standardisieren
- Kategorische Variablen encodieren: Binär-, One-Hot- und Target-Encoding
- Vorbereitete Daten speichern

**3. Übungsaufgabe**
- Eigenständige Anwendung der Schritte auf einen Datensatz

#### Import der Python Bibliotheken


In [ ]:
# Basic packages
import pandas as pd       # Datenmanipulation und -analyse (Tabellenstrukturen)
import numpy as np        # Numerische Operationen, Arrays, Zufallsfunktionen
import matplotlib.pyplot as plt  # Erstellung von Diagrammen und Visualisierungen

---
---

# 1. Importieren der Daten.
## 1.1 Datensätze einladen
Wir verwenden zwei Datensätze
- Nitratdatensatz (Nitratmessungen an GW-Messstellen)
- [Corine Landnutzungarten](https://land.copernicus.eu/pan-european/corine-land-cover/clc2018) an den GW-Messstellen


Wir verwenden die Funktion <span style="color:blue">**pandas.read_csv**</span> &rarr; [Hilfe](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) um die Dateien zu importieren.

### 1.1.1 Nitratdatensatz:

In [ ]:
# Nitratdatensatz einlesen
data_NO3 = pd.read_csv(
    'Nitratmessungen.csv',
    sep=',',
    encoding='utf-8',
    index_col=0
)

# Zeige die ersten 10 Zeilen des Datensatzes an
data_NO3.head(10)

In [ ]:
# Ausgabe der Datentypen aller Spalten im DataFrame
# (z.B. int64, float64, bool, datetime64, timedelta, object usw.)
data_NO3.dtypes


<div style="background-color: #fff8dc; border-left: 6px solid #f0ad4e; padding: 10px; margin: 10px 0;">
<b>Hinweis:</b><br><br>
Beim Überprüfen der Datentypen fällt auf, dass mehrere Spalten einen falschen Typ besitzen:
<ul>
<li>Die Spalte <code>Datum</code> liegt aktuell als <code>object</code>-Typ vor, sollte jedoch in das Datetime-Format (<code>datetime64</code>) umgewandelt werden.</li>
<li>Die Spalten mit physikochemischen Eigenschaften sowie der Nitratgehalt sollten in <code>float</code>-Werte konvertiert werden, um numerische Berechnungen korrekt durchführen zu können.</li>
</ul>
Diese notwendigen Anpassungen werden wir im Abschnitt <b>Datenaufbereitung</b> vornehmen.
</div>



### 1.1.2 CORINE-Landnutzung für Messstellen

Die **CORINE Land Cover-Daten** der Europäischen Umweltagentur klassifizieren die Landnutzung bzw. Landbedeckung in Europa.

Sie umfassen unter anderem folgende Hauptklassen:
- **Bebaute Flächen** (Code 1)
- **Landwirtschaftliche Flächen** (Code 2)
- **Wald- und naturnahe Flächen** (Code 3)
- sowie weitere Landnutzungsklassen.

Die Daten liegen als **Raster- oder Vektordaten** vor. In dieser Übung wird eine bereits vorbereitete Datei verwendet.

In der Datei **`Corinedaten.txt`** wurden die CORINE-Klassen an den Standorten der Messstellen bereits extrahiert.

Weitere Informationen zu den verfügbaren **Bändern und Klassen der CORINE-Daten** findet ihr hier:  
&rarr; [CORINE Land Cover Datenbeschreibung (Google Earth Engine)](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_CORINE_V20_100m#bands)

In [ ]:
# CORINE-Datensatz einlesen
data_Corine = pd.read_csv(
    'Corinedaten.txt',
    sep=';',
    decimal=',',
    encoding='utf-8',
    index_col=0
)

# Zeige die ersten 5 Zeilen des Datensatzes an
data_Corine.head()

## 1.2 Zusammenführen und Filtern der Datensätze

### Zusammenführen der Datensätze

Die beiden Datensätze werden mithilfe der Pandas-Funktion <span style="color:blue">**`.merge()`**</span> zusammengeführt.  
&rarr; [Hilfe zur `.merge()`-Funktion (pandas-Dokumentation)](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

- Die **gemeinsame Spalte** für das Zusammenführen ist **`Grundwassernummer`**.
- Aus dem CORINE-Datensatz wird lediglich die Spalte **`RASTERVALU`** benötigt.
- **`RASTERVALU`** enthält den CORINE-Landnutzungscode am Standort der jeweiligen Messstelle.

Weitere Informationen zu den verfügbaren **Bändern und Landnutzungsklassen** findet ihr hier:  
&rarr; [CORINE Land Cover Datenbeschreibung (Google Earth Engine)](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_CORINE_V20_100m#bands)

### 1.2.1 Zusammenführen der Datensätze
<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px; margin: 10px 0;">
<b>ℹ️ Info:</b><br><br>
Für die weitere Verarbeitung benötigen wir aus den Corine-Daten nur die Spalte <code>RASTERVALU</code> sowie die <code>GW-Nummer</code> als Schlüssel, um beim Zusammenführen die richtigen Werte zuzuweisen.
</div>


In [ ]:
# Zusammenführen der beiden Datensätze anhand der gemeinsamen Messstellen-ID
# - Aus dem CORINE-Datensatz werden nur "GW_NUMMER" und "RASTERVALU" übernommen.
# - Inner Join: Nur Messstellen, die in beiden Datensätzen vorkommen, bleiben erhalten.

joined_data = data_NO3.merge(
    data_Corine[["GW_NUMMER", "RASTERVALU"]],
    how="inner",
    left_on="GW-Nummer",
    right_on="GW_NUMMER"
)

# Doppelte Schlüsselspalte aus dem CORINE-Datensatz entfernen
joined_data = joined_data.drop(columns=["GW_NUMMER"])

# Vorschau auf den zusammengeführten Datensatz
joined_data.head()

In [ ]:
# Gebe Datentypen für die einzelnen Spalten des zusammengeführten Dataframes aus
joined_data.dtypes

### 1.2.2 Filtern der Datensätze
### Auswahl relevanter Spalten für die weitere Verarbeitung

Für die weitere Verarbeitung benötigen wir nur noch die folgenden Spalten:

- **Messstelle**
- **GW-Nummer**
- **Datum**
- **NO3** (Nitratgehalt)
- **O2** (Sauerstoffgehalt)
- **RASTERVALU** (Corine Land Cover Klassifikation)
- **Hydrogeologie**

In [ ]:
# Ausgewählte Spalten übernehmen und eine Kopie erstellen
selected_columns = ["Messstelle", "GW-Nummer", "Datum", "NO3 [mg/l]", "O2 [mg/l]", "RASTERVALU", "HYDROGEOL3"]

data = joined_data[selected_columns].copy()

# Datum in ein Datetime-Objekt umwandeln
data["Datum"] = pd.to_datetime(
    data["Datum"],
    format="%d.%m.%Y %H:%M",
    errors="coerce"
)

# Vorschau auf den ausgewählten Datensatz
data.head()

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin: 10px 0;">
<b>Vorsicht:</b><br><br>
Normalerweise könnten wir die Werte einfach mit <code>.astype(float)</code> umwandeln.  
Wenn ihr das versucht, werdet ihr jedoch eine Fehlermeldung erhalten.  
Der Grund: Die Zahlen verwenden ein **Komma** anstelle eines **Punktes** als Dezimaltrennzeichen.
<br><br>
Daher müssen wir vor der Umwandlung das Komma durch einen Punkt ersetzen.<br><br>
Es lohnt sich, die betroffenen Spalten genauer zu betrachten, bevor wir die Konvertierung durchführen.
</div>


---
---

# 2. Datenaufbereitung

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px; margin: 10px 0;">
<b>Hinweis:</b><br><br>
Bevor ihr Manipulationen an einem DataFrame durchführt, kann es hilfreich sein, eine <b>Kopie</b> des DataFrames zu erstellen,  
um mögliche Fehlermeldungen zu vermeiden – insbesondere wenn ihr Zellen mehrmals ausführt.<br><br>
Dies könnt ihr mit folgendem Befehl tun:
<pre><code>df = data.copy()</code></pre>
Auf diese Weise müsst ihr nicht das gesamte Skript erneut ausführen, sondern könnt einfach ab dieser Zelle weiterarbeiten.
</div>


In [ ]:
# Erstelle eine Kopie des DataFrame
df= data.copy()

Spalte `"NO3 [mg/l]"`

In [ ]:
# Sortiere die Werte der Spalte "NO3 [mg/l]" in aufsteigender Reihenfolge
# Hinweis: Der Datensatz enthält Werte unterhalb der Nachweisgrenze (NWG)
df["NO3 [mg/l]"].sort_values()

> **Vorsicht:** Wir sehen hier, dass es Werte gibt, die unterhalb der Nachweisgrenze liegen. Diese wurden mit "<" gekennzeichnet. Da "Pandas" nicht mit "<" arbeiten kann, müssen wir a) die Zeichen entfernen und b) überlegen, was wir mit den Werten unterhalb der Nachweisgrenze machen.


Spalte `"O2 [mg/l]"`

In [ ]:
# Sortiere die Werte der Spalte in aufsteigender Reihenfolge
df["O2 [mg/l]"].sort_values() #--> beeinhaltet NAN Werte

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin: 10px 0;">
<b>Vorsicht:</b><br><br>
Wir haben festgestellt, dass <b>NaN-Werte</b> im Datensatz vorhanden sind.  
Möglicherweise wurden bestimmte Parameter nicht gemessen.  
<br><br>
In vielen Fällen müssen fehlende Werte geschätzt oder <b>imputiert</b> werden.  
Eine allgemein gültige Regel, wie viele Werte maximal geschätzt werden dürfen, gibt es jedoch nicht.  
Dies hängt unter anderem ab von:
<ul>
<li>der Art der Daten,</li>
<li>der Qualität der vorhandenen Daten,</li>
<li>der gewählten Analysemethode</li>
<li>und dem konkreten Anwendungsfall.</li>
</ul>

<b>Praxisempfehlung:</b>  
Oft wird empfohlen, dass nicht mehr als <b>5–10%</b> der Daten imputiert werden sollten,  
um die Zuverlässigkeit der Ergebnisse zu gewährleisten.
</div>


In [ ]:
# FYI: Zähle die Anzahl von "<"-Symbolen und NaN-Werten in ausgewählten Spalten

# Spaltennamen definieren
columns = ['O2 [mg/l]', 'NO3 [mg/l]']

# Schleife durch jede angegebene Spalte
for col in columns:
    # Anzahl der "<"-Zeichen zählen (nur in Strings sinnvoll)
    count_lessthan = df[col].astype(str).str.contains("<", regex=False).sum()
    
    # Anzahl der NaN-Werte zählen
    count_nan = df[col].isna().sum()
    
    # Gesamtzahl der Einträge (inklusive NaN)
    total_entries = len(df)
    
    # Prozentsätze berechnen
    percent_lessthan = round((count_lessthan / total_entries) * 100, 2)
    percent_nan = round((count_nan / total_entries) * 100, 2)
    
    # Strukturierte Ausgabe
    print(f"🔹 Spalte: '{col}'")
    print(f"   - '<'-Zeichen: {count_lessthan} Einträge ({percent_lessthan}%)")
    print(f"   - NaN-Werte: {count_nan} Einträge ({percent_nan}%)\n")


### 2.1 Ersetzen von Werten unterhalb der Nachweisgrenze (NWG)

Die **Nachweisgrenze (NWG)** ist der kleinste messbare Wert eines Analyten in einem bestimmten System oder Labor.  
Werte unterhalb der NWG gelten als **nicht signifikant**.

In diesem Schritt werden solche Werte durch **`0,5 × NWG`** ersetzt,  
um eine grobe Schätzung des tatsächlichen Wertes zu erhalten – basierend auf der Annahme,  
dass der wahre Wert irgendwo zwischen 0 und der Nachweisgrenze liegt.

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin-top: 15px;">
<b> Hinweis:</b><br><br>
Die Behandlung von Werten unterhalb der NWG ist nicht in allen Fällen sinnvoll.  
Je nach Datentyp, Analysemethode und wissenschaftlichem Kontext kann es geeigneter sein,  
fehlende Werte gezielt auszuschließen oder alternative Schätzverfahren einzusetzen.
</div>

In [ ]:
def ersetze_unter_NWG(wert):
    """
    Ersetzt Werte unterhalb der Nachweisgrenze (NWG) durch 0.5 * NWG.
    Beispiele:
    - "<1,0"  -> 0.5
    - "2,5"   -> 2.5
    - 3.0     -> 3.0
    - NaN     -> NaN
    """

    # Fehlende Werte unverändert lassen
    if pd.isna(wert):
        return np.nan
    # Strings bereinigen
    if isinstance(wert, str):
        wert = wert.strip()

        if wert == "":
            return np.nan

        if "<" in wert:
            wert = wert.replace("<", "").replace(",", ".")
            return float(wert) / 2

        return float(wert.replace(",", "."))

    # Numerische Werte unverändert zurückgeben
    return wert

### Anwenden der Funktion <span style="color:green"><b>ersetze_unter_NWG</b></span> auf die Messdaten

Wir wenden die <span style="color:green"><b>ersetze_unter_NWG</b></span>-Funktion  
auf die Spalten `'NO3 [mg/l]'` und `'O2 [mg/l]'` an,  
um daraus die neuen Spalten **`NO3`** und **`O2`** zu erstellen.

Verwendete Funktionen:
- <span style="color:blue"><b>.apply</b></span>-Methode &rarr; [Hilfe zur `.apply`-Methode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)
- <span style="color:blue"><b>.drop</b></span>-Methode &rarr; [Hilfe zur `.drop`-Methode](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)

Anschließend werden die ursprünglichen Spalten (`'NO3 [mg/l]'` und `'O2 [mg/l]'`) entfernt,  
um den DataFrame übersichtlicher zu gestalten.


In [ ]:
# Erstellen neuer Spalten "O2" und "NO3" durch Anwendung der ersetze_unter_NWG-Funktion

# Wandle die Spalte "O2 [mg/l]" um und speichere das Ergebnis in einer neuen Spalte "O2"
df['O2'] = df['O2 [mg/l]'].apply(ersetze_unter_NWG)

# Wandle die Spalte "NO3 [mg/l]" um und speichere das Ergebnis in einer neuen Spalte "NO3"
df['NO3'] = df['NO3 [mg/l]'].apply(ersetze_unter_NWG)

# Löschen der alten Spalten "O2 [mg/l]" und "NO3 [mg/l]", um den DataFrame aufzuräumen
df.drop(columns=['O2 [mg/l]', 'NO3 [mg/l]'], inplace=True)

# Vorschau: Zeige die ersten fünf Zeilen des aktualisierten DataFrames
df.head()


### 2.2 Ersetzen von fehlenden Werten (NULL-Values)

Datensätze sind häufig unvollständig und enthalten fehlende Werte, sogenannte **NULL-Werte** oder **NaN-Einträge**.  
Diese können verschiedene Ursachen haben, etwa unvollständige Erfassungen, Sensorausfälle oder fehlerhafte Datenübertragungen.

Fehlende Werte sind problematisch, da viele statistische Methoden und Machine-Learning-Algorithmen nicht direkt mit ihnen umgehen können.

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px; margin-top: 15px;">
<b>Wichtig:</b><br><br>
Es ist entscheidend, den Unterschied zwischen <code>0</code> und <code>NULL</code> zu verstehen:<br>
- <code>0</code> steht für eine tatsächlich gemessene Null.<br>
- <code>NULL</code> bedeutet, dass kein Messwert vorhanden ist.
</div>

---

Das vollständige Entfernen von Zeilen mit fehlenden Werten ist nur sinnvoll, wenn deren Anteil sehr gering ist.  
Andernfalls kann es zu einem erheblichen Informationsverlust kommen.

In den meisten Fällen werden fehlende Werte daher durch **Imputation** ersetzt.

Typische Imputationsmethoden sind:
- Ersetzen durch **Mittelwert**, **Median** oder **Modalwert (Mode)**
- Vorhersage fehlender Werte mithilfe von **Machine-Learning-Modellen**

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin-top: 15px;">
<b>Hinweis:</b><br><br>
Die Wahl der Imputationsmethode sollte stets sorgfältig erfolgen und validiert werden,  
da sie die Ergebnisse maßgeblich beeinflussen kann.
</div>

#### Typische Methoden zur Kontrolle fehlender Werte

- **`isnull().sum()`**  
  → Zählt die Anzahl der fehlenden Werte (`NaN`) pro Spalte.

- **`isnull().mean()`**  
  → Berechnet den Anteil fehlender Werte pro Spalte (in Prozent, wenn mit 100 multipliziert).


In [ ]:
# Kontrolle: Ausgabe der Anzahl fehlender Werte pro Spalte
df.isnull().sum()


#(df.isnull().sum()/len(df))*100

### 2.2.1 Ersetzen von NaN-Werten in numerischen Spalten

Für die numerische Spalte **`O2`** werden die fehlenden Werte durch den Mittelwert ersetzt.

Dies geschieht mithilfe des <span style="color:blue"><b>`fillna()`</b></span>-Befehls für Pandas-DataFrames.


<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px; margin-top: 15px;">
<b>Zur Info:</b><br><br>
Falls fehlende Werte durch spezielle Codes wie beispielsweise <code>-999</code> dargestellt sind,  
können Sie diese mit dem Befehl <code>replace('alter_Wert', 'neuer_Wert')</code> ersetzen.  
<br><br>
Dieser Befehl ersetzt alle Vorkommen des alten Werts durch den neuen Wert im gesamten DataFrame.
</div>


In [ ]:
# Ersetzen fehlender Werte (NaN) in der Spalte "O2" durch den Mittelwert
df["O2"] = df["O2"].fillna(df["O2"].mean())

# Kontrolle: Ausgabe der Anzahl fehlender Werte pro Spalte
df.isnull().sum()

### 2.2.2 Ersetzen von NaN-Werten in kategorischen Spalten

In kategorischen Spalten werden fehlende Werte (`NaN`) typischerweise durch den am häufigsten vorkommenden Wert ersetzt,  
den sogenannten **Modus**.

Der Modus ist der Wert, der in der Spalte am häufigsten vorkommt und stellt damit eine sinnvolle Schätzung für fehlende Werte dar.


In [ ]:
# Bestimmung des Modus (häufigster Wert) der Spalte "HYDROGEOL3"
hy_mode = df['HYDROGEOL3'].mode()

# Ausgabe der Häufigkeitsverteilung der Werte in der Spalte "HYDROGEOL3"
print("Häufigkeitsverteilung von 'HYDROGEOL3':")
print(df['HYDROGEOL3'].value_counts())

# Ausgabe des Modus
print("\nModus der Spalte 'HYDROGEOL3':", hy_mode.iloc[0])


In [ ]:
# Ersetzen fehlender Werte in "HYDROGEOL3" durch den Modus der Spalte
df['Hydrogeologie'] = df['HYDROGEOL3'].fillna(hy_mode.iloc[0])
# Entfernen der alten Spalte "HYDROGEOL3" aus dem Datensatz
df.drop(columns='HYDROGEOL3', inplace=True)

#### Kontrolle

In [ ]:
# Kontrolle: Ausgabe der Anzahl fehlender Werte pro Spalte
df.isnull().sum()

In [ ]:
df


## 2.3 Feature-Aggregation

Wir möchten die Daten **pro Messstelle** aggregieren. Dadurch entsteht ein Datensatz, in dem jede Messstelle nur noch einmal vorkommt.

- Numerische Merkmale, z. B. **NO3** und **O2**, werden durch den **Mittelwert** je Messstelle zusammengefasst.
- Kategorische Merkmale, z. B. **HYDROGEOL3** oder **RASTERVALU**, werden durch den **Modus** je Messstelle zusammengefasst.

Zur Umsetzung verwenden wir:

- **`groupby()`**, um die Daten nach Messstellen zu gruppieren.
- **`agg()`**, um unterschiedliche Aggregationsmethoden auf verschiedene Spalten anzuwenden.

Nach der Aggregation:
- wird der Index mit **`reset_index()`** zurückgesetzt,
- und die aggregierten Daten werden wieder zu einem gemeinsamen DataFrame zusammengeführt.

In [ ]:
# Aggregation der numerischen Spalten ("NO3", "O2") mit dem Mittelwert
# und der kategorialen Spalten ("Hydrogeologie", "RASTERVALU") mit dem Modus
GWM = df.groupby('GW-Nummer').agg({
    'NO3': 'mean',
    'O2': 'mean',
    'Hydrogeologie': pd.Series.mode,
    'RASTERVALU': pd.Series.mode
})

# Zurücksetzen des Index, damit "GW-Nummer" wieder eine normale Spalte ist
GWM.reset_index(inplace=True)

# Umbenennen der Spaltennamen für die kategorialen Merkmale (optional, aber für Klarheit empfohlen)
GWM.rename(columns={
    'Hydrogeologie': 'Modus_Hydrogeologie',
    'RASTERVALU': 'Modus_RASTERVALU'
}, inplace=True)

# Ausgabe des aggregierten DataFrames
GWM


## 2.4 Train/Test Split

Das Aufteilen des Datensatzes erfolgt mit der Funktion  
<span style="color:blue"><b><a href="https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html">train_test_split()</a></b></span>  
aus dem Paket **scikit-learn**.

Mit folgenden Parametern lässt sich das Split-Verhalten steuern:
- **`test_size`**: Legt das Verhältnis zwischen Trainings- und Testdaten fest.
- **`random_state`**: Sorgt für Reproduzierbarkeit, indem die Zufallsauswahl kontrolliert wird.

### Warum wird ein Train/Test Split durchgeführt?

Beim maschinellen Lernen ist es wichtig, die **Generalisierungsfähigkeit** eines Modells zu überprüfen.  
Das bedeutet: Wir wollen wissen, wie gut das Modell auf **neue, unbekannte Daten** reagiert – nicht nur auf die Daten, mit denen es trainiert wurde.

Daher teilen wir den Datensatz auf:
- **Trainingsdaten** (`train set`): Diese Daten werden genutzt, um das Modell zu trainieren.
- **Testdaten** (`test set`): Diese Daten bleiben während des Trainings unberührt und werden erst danach verwendet, um die Modellleistung objektiv zu bewerten.

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin-top: 15px;">
<b>Hinweis:</b><br><br>
Der Train/Test-Split verhindert Overfitting nicht direkt, sondern hilft dabei, Overfitting zu erkennen und die Modellqualität realistischer einzuschätzen.
</div>

Eine typische Aufteilung ist:
- 70–80 % Trainingsdaten
- 20–30 % Testdaten

Je nach Anwendungsfall kann das Verhältnis angepasst werden.

In [ ]:
from sklearn.model_selection import train_test_split

# Aufteilen des GWM-Datensatzes in Training- und Testset
# test_size=0.2 bedeutet: 20 % der Daten werden als Testdaten verwendet
train, test = train_test_split(GWM, test_size=0.2, random_state=10)

# Ausgabe der Größe der Trainings- und Testdaten
print('Größe der Trainingsdaten:', train.shape)
print('Größe der Testdaten:', test.shape)


## 2.5 Skalieren

Das Skalieren von Daten vor der Verwendung in einem Machine-Learning-Modell hilft dabei:
- die Leistung des Modells zu verbessern,
- die Berechnung zu stabilisieren,
- und Verzerrungen durch unterschiedlich große Wertebereiche zu vermeiden.

Es gibt verschiedene Skalierungsmethoden, unter anderem:
- **Min-Max-Skalierung** (Transformation auf einen festen Bereich, z. B. [0,1]),
- **Standardisierung** (Zentrierung auf Mittelwert 0 und Standardabweichung 1),
- **Logarithmische Skalierung** (für stark verzerrte Datenverteilungen).

> Die beste Methode hängt von den Eigenschaften der Daten und dem eingesetzten Modell ab.

### 2.5.1 Min-Max-Skalierung

Bei der Min-Max-Skalierung werden Daten auf einen gemeinsamen Wertebereich gebracht, häufig auf den Bereich zwischen 0 und 1.

#### 2.5.1.1 Händisches Skalieren mit Hilfe der Formel:

$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$

Dabei gilt:
- $X$ ist der ursprüngliche Wert,
- $X_{min}$ ist der kleinste Wert der Spalte,
- $X_{max}$ ist der größte Wert der Spalte,
- $X_{norm}$ ist der skalierte Wert.

Der kleinste Wert erhält dadurch den Wert 0, der größte Wert den Wert 1.

In [ ]:
# Minimum und Spannweite der Spalte "NO3" im Trainingsdatensatz berechnen
X_min = train["NO3"].min()
X_range = train["NO3"].max() - train["NO3"].min()

# Min-Max-Skalierung der Spalte "NO3" im Trainingsdatensatz
train["NO3_norm"] = (train["NO3"] - X_min) / X_range

# Min-Max-Skalierung der Spalte "NO3" im Testdatensatz
# Achtung: Die Skalierung basiert nur auf den Trainingsdaten, um Datenleckage zu vermeiden.
test["NO3_norm"] = (test["NO3"] - X_min) / X_range

# Statistische Zusammenfassung der skalierten "NO3"-Spalte im Trainingsdatensatz
print(train["NO3_norm"].describe()[["min", "max", "mean"]])

#### 2.5.1.2 Verwendung der vorhandenen Funktion `MinMaxScaler` aus sklearn

Anstatt die Min-Max-Skalierung manuell durchzuführen, kann die vorhandene Funktion **`MinMaxScaler`** aus **scikit-learn** verwendet werden.

Dabei folgt das Vorgehen einem festen Ablauf:
- Zuerst wird ein eigenes **Scaler-Objekt** erstellt.
- Dieses Objekt wird anschließend an den Trainingsdaten **gefittet** (`fit()`).
- Danach können die Trainings- und Testdaten basierend auf diesem Fit **transformiert** werden (`transform()`).

<div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 10px; margin-top: 15px;">
<b>Wichtig:</b><br><br>
Der Scaler wird ausschließlich an den Trainingsdaten gefittet, um Datenleckage zu vermeiden.  
Die Testdaten werden nur transformiert, aber niemals beim Fit berücksichtigt.
</div>

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# MinMaxScaler-Objekt erstellen und nur an den Trainingsdaten anpassen
scaler = MinMaxScaler()
scaler.fit(train[["NO3"]])

# NO3 im Trainings- und Testdatensatz skalieren und als neue Spalte speichern
train["NO3_norm"] = scaler.transform(train[["NO3"]])
test["NO3_norm"] = scaler.transform(test[["NO3"]])

# Kontrolle der skalierten Werte im Trainingsdatensatz
print(train["NO3_norm"].describe()[["min", "max", "mean"]])

### 2.5.2 Standardisieren

Die **Standardisierung** ist ein Verfahren, bei dem Daten so transformiert werden,  
dass sie einen **Mittelwert von 0** und eine **Standardabweichung von 1** erhalten.

Die Standardisierung erfolgt durch:
- Subtraktion des Mittelwerts der Daten,
- Division durch die Standardabweichung der Daten.

Dadurch werden die Daten **zentriert** (auf Mittelwert 0) und **skaliert** (auf Standardabweichung 1).

Die Standardisierung kann helfen:
- Merkmale mit unterschiedlichen Wertebereichen vergleichbarer zu machen,
- und die Modellleistung zu verbessern, insbesondere bei Verfahren, die empfindlich gegenüber unterschiedlichen Skalen sind.

#### 2.5.2.1 Händisches Standardisieren mit Hilfe der Formel

Die Standardisierung eines Werts \( X \) erfolgt über folgende Formel:

$X_{\text{std}} = \frac{X - \text{mean}(X)}{\text{std}(X)}$

In [ ]:
# Berechnen von Mittelwert und Standardabweichung der NO3-Spalte im Trainingsdatensatz
X_mean = train["NO3"].mean()
X_std = train["NO3"].std()

# Standardisierung der NO3-Spalte in Trainings- und Testdaten
# Achtung: Mittelwert und Standardabweichung werden nur aus den Trainingsdaten berechnet,
# um Datenleckage zu vermeiden.
train["NO3_std"] = (train["NO3"] - X_mean) / X_std
test["NO3_std"] = (test["NO3"] - X_mean) / X_std

# Kontrolle der standardisierten Werte im Trainingsdatensatz
print('mean(X): ', train['NO3_std'].mean())
print('std(X): ', train['NO3_std'].std())

#### 2.5.2.2 Vorhandene Funktion von sklearn "StandardScaler"

In [ ]:
from sklearn.preprocessing import StandardScaler

# StandardScaler-Objekt erstellen und nur an den Trainingsdaten anpassen
scaler = StandardScaler()
scaler.fit(train[["NO3"]])

# NO3 im Trainings- und Testdatensatz standardisieren und als neue Spalte speichern
train["NO3_std"] = scaler.transform(train[["NO3"]])
test["NO3_std"] = scaler.transform(test[["NO3"]])

# Kontrolle der standardisierten Werte im Trainingsdatensatz
print("mean(X): ", train["NO3_std"].mean())
print("std(X): ", train["NO3_std"].std(ddof=0))

### 2.5.3 Robustes Skalieren (RobustScaler)

Der **RobustScaler** ist eine Skalierungsmethode, die weniger empfindlich gegenüber **Ausreißern** ist als die Standardisierung.

Im Gegensatz zum StandardScaler verwendet er:
- den **Median** anstelle des Mittelwerts,
- und den **Interquartilsabstand (IQR)** anstelle der Standardabweichung.

Eigenschaften:
- Der Median der transformierten Daten ist etwa **0**
- Der IQR der transformierten Daten ist etwa **1**
- Ausreißer haben weniger Einfluss auf die Skalierung

Diese Methode ist besonders geeignet für Datensätze mit:
- starken Ausreißern
- schiefen Verteilungen



In [ ]:
from sklearn.preprocessing import RobustScaler

# RobustScaler-Objekt erstellen und nur an den Trainingsdaten anpassen
scaler = RobustScaler()
scaler.fit(train[["NO3"]])

# NO3 im Trainings- und Testdatensatz robust skalieren und als neue Spalte speichern
train["NO3_rb"] = scaler.transform(train[["NO3"]])
test["NO3_rb"] = scaler.transform(test[["NO3"]])

# Kontrolle der robust skalierten Werte im Trainingsdatensatz
print("median(X): ", train["NO3_rb"].median())
print("IQR(X): ", train["NO3_rb"].quantile(0.75) - train["NO3_rb"].quantile(0.25))

### 2.6.1 Vereinfachtes binäres Encodieren

Beim binären Encodieren wird eine Kategorie in zwei mögliche Werte überführt, meist:

- **1** = trifft zu
- **0** = trifft nicht zu

**Beispiel: CORINE-Landnutzungscodes**

Der dreistellige CORINE-Code beschreibt die Landnutzung hierarchisch:
- **1XX** = Artificial Surfaces
- **2XX** = Agricultural Areas
- **3XX** = Forest and Seminatural Areas
- **4XX** = Wetlands
- **5XX** = Water Bodies

Dabei steht die erste Ziffer für die oberste Landnutzungsklasse.

Im Folgenden wird ein binäres Merkmal erstellt, das angibt, ob eine Messstelle in einer landwirtschaftlich geprägten Fläche liegt:

- **1** = CORINE-Code beginnt mit 2 → Agricultural Area
- **0** = andere CORINE-Hauptklasse

In [ ]:
# Binäres Encoding der Landnutzung: Artificial Surfaces (1XX-Codes) vs. andere Klassen

for dataset in [train, test]:
    # Erste Ziffer des CORINE-Codes extrahieren
    dataset["Corine"] = dataset["Modus_RASTERVALU"].astype(str).str[0].astype(int)
    
    # Binäre Spalte erstellen:
    # 1 = Artificial Surfaces, 0 = andere Landnutzungsklassen
    dataset["Artificial_Surface"] = (dataset["Corine"] == 1).astype(int)
    
    # Ursprüngliche CORINE-Spalte entfernen
    dataset.drop(columns=["Modus_RASTERVALU"], inplace=True)


### 2.6.2 One-Hot-Encoding

One-Hot-Encoding ist ein Verfahren zur Darstellung von **kategorischen** oder **nominalen Daten** (Daten ohne natürliche Rangfolge) als numerische Vektoren.

Dabei wird:
- jeder eindeutigen Kategorie eine eigene neue Spalte zugewiesen,
- und innerhalb dieser Spalte wird für jede Beobachtung ein **Binärwert** gesetzt (1 oder 0).

Das bedeutet:
- Eine "1" zeigt an, dass eine Beobachtung zu dieser Kategorie gehört.
- Eine "0" zeigt an, dass sie nicht zu dieser Kategorie gehört.

One-Hot-Encoding wird insbesondere für **nominale Merkmale** eingesetzt,  
um sie in eine Form zu bringen, die von maschinellen Lernmodellen verarbeitet werden kann,  
ohne eine künstliche Rangordnung zu erzeugen.


Beispiel für One-Hot Encoding:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# OneHotEncoder-Objekt erstellen
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

# Encoder nur an den Trainingsdaten fitten
encoder.fit(train[["Corine"]])

# Train- und Testdaten transformieren
OHE_train = encoder.transform(train[["Corine"]])
OHE_test = encoder.transform(test[["Corine"]])

# In DataFrames umwandeln
OHE_train = pd.DataFrame(
    OHE_train,
    columns=encoder.get_feature_names_out(["Corine"]),
    index=train.index
)

OHE_test = pd.DataFrame(
    OHE_test,
    columns=encoder.get_feature_names_out(["Corine"]),
    index=test.index
)

# Ausgabe der One-Hot-kodierten Trainingsdaten
OHE_test.head()

### 2.6.3 Target Encoding

Beim **Target Encoding** wird der Mittelwert der Zielvariable (Target) für jede Kategorie berechnet und anschließend als numerischer Wert für diese Kategorie verwendet.

Das Verfahren läuft typischerweise so ab:
- Für jede Ausprägung einer kategorialen Variable wird der Durchschnittswert der Zielvariable im Trainingsdatensatz ermittelt.
- Dieser Durchschnitt ersetzt die ursprüngliche Kategorie.

Beispiel:
- Kategorie A → durchschnittlicher Target-Wert = 10
- Kategorie B → durchschnittlicher Target-Wert = 5

⇒ A wird zu 10, B wird zu 5

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin-top: 15px;">
<b>Wichtig:</b><br><br>
Target Encoding muss <b>nach</b> dem Train/Test-Split durchgeführt werden,  
damit keine Informationen aus den Testdaten in das Training einfließen (Vermeidung von Data Leakage).

Zusätzlich besteht die Gefahr von <b>Overfitting</b>, da die Zielvariable direkt verwendet wird.  
In der Praxis werden daher oft Techniken wie Cross-Validation oder Glättung (Smoothing) eingesetzt.
</div>

In [ ]:
from category_encoders.target_encoder import TargetEncoder

# TargetEncoder-Objekt erstellen
encoder = TargetEncoder()

# Encoder nur auf den Trainingsdaten fitten
train["Target_Hy"] = encoder.fit_transform(
    train[["Modus_Hydrogeologie"]],
    train["NO3"]
)

# Testdaten nur transformieren
test["Target_Hy"] = encoder.transform(
    test[["Modus_Hydrogeologie"]]
)

# Kontrolle
train.head()


## 2.7 Speichern der Ergebnisse

Um einen Datensatz zu speichern, ist es sinnvoll, ihn als **CSV-Datei** zu exportieren.

Das Exportieren eines DataFrames erfolgt mit dem Befehl  
<span style="color:blue"><b><a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html">.to_csv()</a></b></span>  
aus der Pandas-Bibliothek.

Damit kann der bearbeitete Datensatz dauerhaft gespeichert und später einfach wieder eingelesen werden.


In [ ]:

# Speichern des Trainingsdatensatzes als CSV-Datei im Ordner "Termin_1"
# Hinweis: Ohne Angabe von "index=False" wird der Index als zusätzliche Spalte mitgespeichert
train.to_csv('results.csv', index=False)


# 3. Übung: Eigene Datenaufbereitung

Bereite den Datensatz **"Nitratmessungen_aufgabe.csv"** eigenständig auf:

### Aufgaben:

1. Ersetze alle Werte unterhalb der Nachweisgrenze (NWG).
2. Ersetze alle fehlenden Werte (`NULL`-Values).
3. Aggregiere die Daten für jede Messstelle.
4. Splitte den Datensatz im Verhältnis 85 % Training : 15 % Test.
5. Standardisiere die numerischen Variablen.
6. Führe ein binäres Encoding der Landnutzung durch:
   - **Landwirtschaft = 2XX** ➔ `1`
   - **Alle anderen Klassen** ➔ `0`
7. Target-Encode die Hydrogeologie basierend auf der Nitratkonzentration.
8. Speichere deine Ergebnisse.
9. Fertig!

---

<div style="background-color: #fff3cd; border-left: 6px solid #ffa502; padding: 10px; margin-top: 15px;">
<b>Tipps:</b><br><br>
- Schaue dir die Features zur <b>Sauerstoff-Konzentration</b> und <b>Hydrogeologie</b> genau an.<br>
- Fehlende Werte sind hier nicht immer als klassische <code>NULL</code> oder <code>NaN</code> dargestellt.<br>
- Nutze zur Analyse:<br>
  - <code>.describe()</code> → Statistische Übersicht<br>
  - <code>.unique()</code> → Alle vorkommenden Werte
</div>
